# Módulo 2: Estadística y Visualización de Datos
- **Sesión:** 05 — Mapas y geovisualización
- **Cuaderno del alumno**
- **Fecha:** 06/12/2025
- **Actualización:** 22/08/2026

## Objetivos de la sesión
1. Comprender los fundamentos de la geovisualización: coordenadas, códigos de país y tipos de mapas temáticos.
2. Construir **mapas coropléticos** con Plotly Express a partir de DataFrames.
3. Construir **mapas interactivos** con Folium: marcadores, popups y círculos proporcionales.
4. Desarrollar criterio sobre cuándo un mapa comunica mejor que un gráfico tradicional — y cuándo no.

## Requisitos
```
pip install plotly folium
```
> ⚠️ Los mapas de esta sesión requieren **conexión a internet** para renderizarse (Plotly carga las geometrías y Folium los mosaicos del mapa base).

> 📝 Los ejercicios están **propuestos** a lo largo del notebook, con una celda vacía para su solución.

## 1. Fundamentos de geovisualización

### Teoría

Un dato es **georreferenciable** cuando puede asociarse a una ubicación. Las dos formas más comunes:

- **Coordenadas geográficas**: latitud $\varphi \in [-90°, 90°]$ (negativa al sur del ecuador — Lima está en $\varphi \approx -12°$) y longitud $\lambda \in [-180°, 180°]$ (negativa al oeste de Greenwich — Lima en $\lambda \approx -77°$). Sirven para ubicar **puntos**.
- **Códigos de país ISO 3166**: identificadores estandarizados de 3 letras (`PER`, `CHL`, `BRA`). Sirven para colorear **regiones completas** sin necesitar sus geometrías: las librerías ya las traen.

Tipos de mapa temático que veremos:

| Tipo de mapa | Qué representa | Herramienta en esta sesión |
|---|---|---|
| **Coroplético** | Una variable numérica como **color de cada región** | Plotly Express (`px.choropleth`) |
| **De marcadores** | **Puntos de interés** con información al hacer clic | Folium (`folium.Marker`) |
| **De símbolos proporcionales** | Una magnitud como **tamaño de un círculo** en cada punto | Folium (`folium.CircleMarker`) |

⚠️ Una advertencia que retomaremos al final: en un coroplético el ojo pondera el color por el **área** de la región, así que los países grandes dominan visualmente aunque su valor no sea el mayor.

## 2. Mapas coropléticos con Plotly Express

### Teoría

**Plotly Express** (`px`) es una librería de gráficos **interactivos** (zoom, hover con datos). Para un coroplético necesita tres cosas:

| Parámetro | Qué recibe |
|---|---|
| `locations` | Columna con los códigos ISO-3 de cada país |
| `color` | Columna numérica que define el color |
| `hover_name` / `hover_data` | Qué mostrar al pasar el mouse |
| `color_continuous_scale` | Paleta (`"Viridis"`, `"YlOrRd"`, `"RdYlGn"`...) |
| `scope` | Recorte del mundo (`"south america"`, `"world"`...) |

Nuestro archivo del FMI trae **nombres** de países, no códigos ISO-3, así que primero construimos la columna con un diccionario de mapeo.

> Nota: para proyectos grandes existe la librería `pycountry`, que convierte nombres a códigos automáticamente; aquí usamos un diccionario explícito para ver el mecanismo.

### Laboratorio

In [2]:
import pandas as pd
import plotly.express as px

df_fmi = pd.read_csv("dataset/fmi/WEO_Data_Latinoamerica_2026.csv",
                     thousands=",", decimal=".", sep=",")
df_fmi.head(3)

,DATASET,SERIES_CODE,OBS_MEASURE,COUNTRY,INDICATOR,FREQUENCY,SCALE,2000,2001,2002,...,2022,2023,2024,2025,2026,2027,2028,2029,2030,2031
0,IMF.RES:WEO(9.0.0),ARG.NGDPD.A,OBS_VALUE,Argentina,"Gross domestic product (GDP), Current prices, ...",Annual,Billions,317.759,300.421,112.458,...,633.520,648.895,637.234,681.485,688.378,703.668,746.843,789.967,833.284,878.051
1,IMF.RES:WEO(9.0.0),BOL.NGDPD.A,OBS_VALUE,Bolivia,"Gross domestic product (GDP), Current prices, ...",Annual,Billions,9.371,9.100,8.895,...,51.331,52.722,55.281,64.250,80.743,NaN,NaN,NaN,NaN,NaN
2,IMF.RES:WEO(9.0.0),COL.NGDPD.A,OBS_VALUE,Colombia,"Gross domestic product (GDP), Current prices, ...",Annual,Billions,99.270,97.606,97.353,...,345.632,366.901,420.504,457.410,539.530,554.384,578.238,604.409,632.137,661.138


In [4]:
# Mapeo nombre -> código ISO-3 (ajuste los nombres a los de su archivo)
iso3 = {
    "Peru": "PER", 
    "Chile": "CHL", 
    "Argentina": "ARG", 
    "Colombia": "COL",
    "Brazil": "BRA", 
    "Mexico": "MEX", 
    "Venezuela": "VEN", 
    "Bolivia": "BOL",
    "Ecuador": "ECU", 
    "Paraguay": "PRY", 
    "Uruguay": "URY",
    "Guatemala": "GTM", 
    "Honduras": "HND", 
    "El Salvador": "SLV",
    "Nicaragua": "NIC", 
    "Costa Rica": "CRI", 
    "Panama": "PAN",
    "Dominican Republic": "DOM", 
    "Cuba": "CUB", 
    "Haiti": "HTI",
}

df_fmi["ISO3"] = df_fmi["COUNTRY"].map(iso3)

# Verificamos qué países quedaron sin código (para completar el diccionario)
print("Sin código ISO:", df_fmi[df_fmi["ISO3"].isna()]["COUNTRY"].tolist())

df_fmi = df_fmi.dropna(subset=["ISO3"])

df_fmi.head()

Sin código ISO: []


,DATASET,SERIES_CODE,OBS_MEASURE,COUNTRY,INDICATOR,FREQUENCY,SCALE,2000,2001,2002,...,2023,2024,2025,2026,2027,2028,2029,2030,2031,ISO3
0,IMF.RES:WEO(9.0.0),ARG.NGDPD.A,OBS_VALUE,Argentina,"Gross domestic product (GDP), Current prices, ...",Annual,Billions,317.759,300.421,112.458,...,648.895,637.234,681.485,688.378,703.668,746.843,789.967,833.284,878.051,ARG
1,IMF.RES:WEO(9.0.0),BOL.NGDPD.A,OBS_VALUE,Bolivia,"Gross domestic product (GDP), Current prices, ...",Annual,Billions,9.371,9.100,8.895,...,52.722,55.281,64.250,80.743,NaN,NaN,NaN,NaN,NaN,BOL
2,IMF.RES:WEO(9.0.0),COL.NGDPD.A,OBS_VALUE,Colombia,"Gross domestic product (GDP), Current prices, ...",Annual,Billions,99.270,97.606,97.353,...,366.901,420.504,457.410,539.530,554.384,578.238,604.409,632.137,661.138,COL
3,IMF.RES:WEO(9.0.0),BRA.NGDPD.A,OBS_VALUE,Brazil,"Gross domestic product (GDP), Current prices, ...",Annual,Billions,655.454,559.982,509.798,...,2191.137,2185.822,2279.918,2635.912,2766.558,2879.420,3037.171,3203.508,3378.877,BRA
4,IMF.RES:WEO(9.0.0),ECU.NGDPD.A,OBS_VALUE,Ecuador,"Gross domestic product (GDP), Current prices, ...",Annual,Billions,17.531,23.127,27.054,...,120.793,123.802,130.321,138.194,144.135,150.136,156.661,163.818,171.302,ECU


In [5]:
# Primer coroplético: PBI 2025
fig = px.choropleth(df_fmi,
                    locations="ISO3",
                    color="2026",
                    hover_name="COUNTRY",
                    color_continuous_scale="Viridis",
                    title="PBI 2026 — Latinoamérica (miles de millones US$)")
fig.show()

El mapa mundial deja mucho espacio vacío; con `scope` recortamos la vista (pruebe pasar el mouse sobre cada país — esa interactividad es lo que Plotly agrega sobre matplotlib):

In [6]:
fig = px.choropleth(df_fmi,
                    locations="ISO3",
                    color="2026",
                    hover_name="COUNTRY",
                    hover_data=["2020", "2026"],
                    color_continuous_scale="Viridis",
                    scope="south america",
                    title="PBI 2026 — Sudamérica")
fig.show()

### Bonus: animación temporal

Si reorganizamos los datos a formato largo (una fila por país-año, con `melt`), el parámetro `animation_frame` genera un control de reproducción año a año:

In [7]:
anhos = [str(a) for a in range(2015, 2026)]

df_largo = df_fmi.melt(id_vars=["COUNTRY", "ISO3"],
                       value_vars=anhos,
                       var_name="Año", value_name="PBI")

fig = px.choropleth(df_largo,
                    locations="ISO3",
                    color="PBI",
                    hover_name="COUNTRY",
                    animation_frame="Año",
                    color_continuous_scale="Viridis",
                    scope="south america",
                    range_color=(0, df_largo["PBI"].max()),   # escala fija entre años
                    title="Evolución del PBI 2015-2025")
fig.show()

### 📝 Ejercicio 1 — Coroplético del PBI 2026

Construya un mapa coroplético de Latinoamérica con el PBI del año **2026** (archivo `WEO_Data_Latinoamerica_2026.csv`), cumpliendo:

1. Usar la columna de códigos ISO-3 construida en clase.
2. Escala de color `"Viridis"` y título apropiado.
3. Limitar la vista a Sudamérica con `scope="south america"`.

In [ ]:
# Escriba su solución aquí


### 📝 Ejercicio 2 — Coroplético del crecimiento del PBI

Calcule la **tasa de crecimiento porcentual** del PBI de cada país entre 2020 y 2025:

$$\Delta\% = \frac{PBI_{2025} - PBI_{2020}}{PBI_{2020}} \cdot 100$$

y grafíquela en un mapa coroplético con la escala divergente `"RdYlGn"` (rojo = menor crecimiento, verde = mayor). Interprete: ¿qué país creció más en el periodo?

In [ ]:
# Escriba su solución aquí


### 📝 Ejercicio 3 — Coroplético del medallero

Con el archivo `medallero_Panamericanos_Santiago2023.csv`, construya un mapa coroplético **mundial** (sin `scope`) donde el color represente el **Total** de medallas de cada país.

> Los nombres de países del medallero están en español; use el diccionario `iso3_medallero` dado abajo para mapearlos a ISO-3 (complete los que falten según su archivo).

In [ ]:
# Escriba su solución aquí


## 3. Mapas interactivos con Folium

### Teoría

**Folium** genera mapas web interactivos (basados en Leaflet.js) directamente desde Python. A diferencia del coroplético, aquí trabajamos con **puntos** definidos por latitud y longitud:

| Elemento | Descripción |
|---|---|
| `folium.Map(location=[lat, lon], zoom_start=n)` | Mapa base centrado en un punto |
| `folium.Marker(location, popup, tooltip, icon)` | Marcador con globo de información al hacer clic |
| `folium.CircleMarker(location, radius, ...)` | Círculo cuyo radio puede codificar una magnitud |
| `.add_to(mapa)` | Agrega el elemento al mapa |

El objeto mapa se muestra escribiéndolo al final de la celda (como un DataFrame).

### Laboratorio: sedes de los Panamericanos Santiago 2023

In [9]:
import folium

# Mapa base centrado en Santiago de Chile
mapa = folium.Map(location=[-33.45, -70.67], zoom_start=9)
mapa

In [10]:
# Sedes principales de los Juegos (nombre, lat, lon)
sedes = [
    ("Estadio Nacional (ceremonias)", -33.4646, -70.6103),
    ("Parque Deportivo Peñalolén",    -33.4780, -70.5250),
    ("Valparaíso (velódromo)",        -33.0472, -71.6127),
    ("Viña del Mar (triatlón)",       -33.0245, -71.5518),
]

mapa = folium.Map(location=[-33.30, -71.00], zoom_start=9)

for nombre, lat, lon in sedes:
    folium.Marker(
        location=[lat, lon],
        popup=nombre,               # al hacer clic
        tooltip="Ver sede"          # al pasar el mouse
    ).add_to(mapa)

mapa

Los íconos admiten color y símbolo (`folium.Icon(color=..., icon=...)`), y `CircleMarker` reemplaza el pin por un círculo cuyo **radio** puede representar una variable — el equivalente en mapas del gráfico de burbujas de la sesión 04:

In [11]:
mapa = folium.Map(location=[-33.30, -71.00], zoom_start=9)

# Marcador destacado para la sede principal
folium.Marker(
    location=[-33.4646, -70.6103],
    popup="Estadio Nacional",
    icon=folium.Icon(color="red", icon="star")
).add_to(mapa)

# Círculo proporcional: aforo aproximado del estadio
folium.CircleMarker(
    location=[-33.4646, -70.6103],
    radius=30,
    color="darkred", fill=True, fill_opacity=0.3,
    popup="Aforo: ~48 mil espectadores"
).add_to(mapa)

mapa

### 📝 Ejercicio 4 — Marcadores de las principales ciudades del Perú

Construya un mapa de **Folium** centrado en el Perú (aprox. `[-9.19, -75.02]`, `zoom_start=5`) con marcadores para Lima, Cusco, Arequipa, Trujillo e Iquitos, cumpliendo:

1. Cada marcador con un `popup` que muestre el nombre de la ciudad y su población aproximada.
2. El marcador de Lima (capital) con ícono de color rojo, el resto en azul.

In [ ]:
# Escriba su solución aquí


### 📝 Ejercicio 5 — Círculos proporcionales del medallero

Sobre un mapa de Folium centrado en América (`[0, -80]`, `zoom_start=3`), dibuje un `folium.CircleMarker` por cada uno de los **8 primeros países** del medallero de Santiago 2023, donde el **radio** del círculo sea proporcional al Total de medallas (por ejemplo `radius=Total/10`).

Use las coordenadas dadas abajo y agregue un popup con país y total.

In [ ]:
# Escriba su solución aquí


### 📝 Ejercicio 6 — ¿Coroplético o barras? Juicio crítico

Observando el mapa coroplético del PBI (laboratorio) y recordando el gráfico de barras del PBI de la sesión 03, responda en esta celda:

1. ¿Qué comunica mejor el mapa que las barras, y viceversa?
2. ¿Qué problema tiene el coroplético cuando los países son de tamaños muy distintos (por ejemplo, Brasil vs. Uruguay)?
3. ¿En qué caso de su especialidad usaría cada uno?

*✏️ Escriba aquí su respuesta...*

## Resumen de la sesión

| Herramienta | Pregunta que responde | Cuándo usarla |
|---|---|---|
| `px.choropleth` | ¿Cómo se distribuye una variable **por regiones**? | Datos por país/región con código ISO |
| `folium.Marker` | ¿**Dónde** están los puntos de interés? | Ubicaciones puntuales con lat/lon |
| `folium.CircleMarker` | ¿Dónde y **cuánto**? | Magnitudes en ubicaciones puntuales |

**Ideas clave:**
- El coroplético necesita códigos estandarizados (ISO-3); los mapas de puntos necesitan lat/lon.
- El área de los países sesga la lectura del coroplético: para magnitudes absolutas, considerar círculos proporcionales.
- Un mapa responde *"¿dónde?"*; para *"¿cuánto y en qué orden?"* las barras siguen siendo superiores.